<a href="https://www.kaggle.com/code/duttaarya/football-transfer-prediction?scriptVersionId=300292575" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/player-scores/players.csv
/kaggle/input/player-scores/competitions.csv
/kaggle/input/player-scores/games.csv
/kaggle/input/player-scores/transfers.csv
/kaggle/input/player-scores/game_events.csv
/kaggle/input/player-scores/club_games.csv
/kaggle/input/player-scores/player_valuations.csv
/kaggle/input/player-scores/game_lineups.csv
/kaggle/input/player-scores/appearances.csv
/kaggle/input/player-scores/clubs.csv


In [2]:
players = pd.read_csv("/kaggle/input/player-scores/players.csv")
valuation = pd.read_csv("/kaggle/input/player-scores/player_valuations.csv")
appearances = pd.read_csv("/kaggle/input/player-scores/appearances.csv")

In [3]:
valuation.head()
valuation.columns
valuation.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478228 entries, 0 to 478227
Data columns (total 6 columns):
 #   Column                               Non-Null Count   Dtype 
---  ------                               --------------   ----- 
 0   player_id                            478228 non-null  int64 
 1   date                                 478228 non-null  object
 2   market_value_in_eur                  478228 non-null  int64 
 3   current_club_name                    478228 non-null  object
 4   current_club_id                      478228 non-null  int64 
 5   player_club_domestic_competition_id  444508 non-null  object
dtypes: int64(3), object(3)
memory usage: 21.9+ MB


In [4]:
valuation["date"] = pd.to_datetime(valuation["date"])
valuation = valuation.sort_values(["player_id","date"])
latest_valuation = valuation.groupby("player_id").tail(1)

latest_valuation.shape

(30068, 6)

In [5]:
players = pd.read_csv("/kaggle/input/player-scores/players.csv")
players.head()

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,right,184.0,NaN,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,NaN,NaN,NaN,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,NaN,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,right,194.0,NaN,IFM,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


In [6]:
players = players.drop(columns = [
    "image_url",
    "url",
    "player_code",
    "first_name",
    "last_name",
    "name",
    "city_of_birth",
    "agent_name",
    "highest_market_value_in_eur"
])
#if shows error after using it for multiple time then it will show error as you can't 
#drop the same columns for 2 timws

In [7]:
players.head()
players.shape

(33199, 14)

In [8]:
df = latest_valuation.merge(players, on ="player_id", how = 'left')
df.shape
df.head()

,player_id,date,market_value_in_eur_x,current_club_name_x,current_club_id_x,player_club_domestic_competition_id,last_season,current_club_id_y,country_of_birth,country_of_citizenship,date_of_birth,sub_position,position,foot,height_in_cm,contract_expiration_date,current_club_domestic_competition_id,current_club_name_y,market_value_in_eur_y
0,10,2016-01-04,1000000,SS Lazio,398,IT1,2015,398,Poland,Germany,1978-06-09 00:00:00,Centre-Forward,Attack,right,184.0,NaN,IT1,Società Sportiva Lazio S.p.A.,1000000.0
1,26,2017-12-28,750000,Borussia Dortmund,16,L1,2017,16,Germany,Germany,1980-08-06 00:00:00,Goalkeeper,Goalkeeper,left,190.0,NaN,L1,Borussia Dortmund,750000.0
2,65,2016-06-21,1000000,PAOK Thessaloniki,1091,GR1,2015,1091,Bulgaria,Bulgaria,1981-01-30 00:00:00,Centre-Forward,Attack,NaN,NaN,NaN,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0
3,77,2016-11-15,200000,FC Goa,506,IT1,2012,506,Brazil,Brazil,1978-05-08 00:00:00,Centre-Back,Defender,NaN,NaN,NaN,IT1,Juventus Football Club,200000.0
4,80,2018-06-05,100000,Bayern Munich,27,L1,2017,27,East Germany (GDR),Germany,1981-03-18 00:00:00,Goalkeeper,Goalkeeper,right,194.0,NaN,L1,FC Bayern München,100000.0


In [9]:
#creating date of birth for age:- 
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"])


In [10]:
df["date_of_birth"].isna().sum()

np.int64(33)

In [11]:
df = df.dropna(subset=["date_of_birth"])
df["date_of_birth"].isna().sum()


np.int64(0)

In [12]:
df["age"] = (pd.to_datetime("today") - df["date_of_birth"]).dt.days / 365
df["age"] = df["age"].astype(int)

In [13]:
df[["date_of_birth", "age"]].head()

,date_of_birth,age
0,1978-06-09,47
1,1980-08-06,45
2,1981-01-30,45
3,1978-05-08,47
4,1981-03-18,44


In [14]:
df = df.drop(columns=["date_of_birth"])

In [15]:
df.columns #checking the name of the column cause after merging the
           # columns they change the name


Index(['player_id', 'date', 'market_value_in_eur_x', 'current_club_name_x',
       'current_club_id_x', 'player_club_domestic_competition_id',
       'last_season', 'current_club_id_y', 'country_of_birth',
       'country_of_citizenship', 'sub_position', 'position', 'foot',
       'height_in_cm', 'contract_expiration_date',
       'current_club_domestic_competition_id', 'current_club_name_y',
       'market_value_in_eur_y', 'age'],
      dtype='object')

In [16]:
x = df[["age","height_in_cm"]]
y = df["market_value_in_eur_x"]

In [17]:
x.head()
y.head()

0    1000000
1     750000
2    1000000
3     200000
4     100000
Name: market_value_in_eur_x, dtype: int64

In [18]:
x.isna().sum()

age                0
height_in_cm    1575
dtype: int64

In [19]:
df = df.dropna(subset=["age", "height_in_cm"])

In [20]:
x = df[["age","height_in_cm"]]
y = df["market_value_in_eur_x"]

In [21]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [22]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [23]:
y_pred = model.predict(x_test)

In [24]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 2352843.129735516
R2 Score: 0.02444342909171371


MAE = Mean Absolute Error

It means:

On average, your model’s prediction is off by ~2.36 million euros.

So if real value = €10M
Model might predict = €7.6M or €12.3M
That is a large error.

this occurs due to low parameters like only height and age

# now taking more 2 parameters (position and foot) 

In [25]:
df["position"].value_counts()


position
Defender      9109
Midfield      8243
Attack        7857
Goalkeeper    3175
Missing         76
Name: count, dtype: int64

In [26]:
df["foot"].value_counts()

foot
right    19341
left      6821
both      1321
Name: count, dtype: int64

In [27]:
#df_encoded.head()

In [28]:
df = df.rename(columns={"market_value_in_eur_x": "market_value"})

In [29]:
df = df.rename(columns={"market_value_in_eur_x": "market_value"})
df = df.drop(columns=["market_value_in_eur_y"])

In [30]:
df_encoded = pd.get_dummies(df, columns=["position", "foot"], drop_first=True)

In [31]:
X = df_encoded.drop(columns=["market_value", "player_id"])
y = df_encoded["market_value"]

In [32]:
X = X.select_dtypes(include=["int64", "float64"]) 
#changing the data type for linear regression function

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [34]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [35]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 2422057.4451395613
R2: 0.07244289063773646


In [36]:
print("market_value" in X.columns)

False
